# Reproducing the BICR Paper

This notebook regenerates every benchmark table and reliability diagram reported in the paper from the bundled `results/SPARROW/` JSONs. It does **not** require GPU access or running any extraction — the JSONs already contain the predictions and labels for every (method, VLM, seed) cell.

Outputs are written to `docs/tables/` (TeX) and `docs/figures_generated/` (PDF). All numbers should match the paper to two decimals.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "results" / "SPARROW").exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print(f"Repo root: {REPO}")

Repo root: /home/rkhanm1/VLMCE/repo


## Table 1 — Cross-VLM pooled main results (8 methods × 4 metrics)

Averaged across 5 seeds within each VLM, then across the 5 VLMs.

In [2]:
from evaluation.analysis.aggregate import load_main_results, cross_vlm_mean
from evaluation.analysis.tables import build_main_table

main_df = cross_vlm_mean(load_main_results())
main_df = main_df.copy()
for col in ["ece", "brier", "aucpr", "auroc"]:
    main_df[col] = (main_df[col] * 100).round(2)
main_df

metric,method,ece,brier,aucpr,auroc
0,P(True),38.35,39.90,73.10,55.55
1,Self-Probing,27.89,29.77,78.61,65.06
2,PE,18.11,25.88,69.53,59.97
3,P(IK),9.36,19.46,86.25,76.55
4,SAPLMA,12.30,21.38,81.55,72.89
5,InternalInspector,8.34,19.90,84.27,74.83
6,CCPS,15.33,27.17,72.75,63.08
7,BICR,7.08,18.43,87.43,78.61


In [3]:
build_main_table(REPO / "docs" / "tables" / "main_results.tex")
print("Wrote docs/tables/main_results.tex")

Wrote docs/tables/main_results.tex


## Table 2 — BICR loss-component ablation

Full / −Brier / −Rank / BCE-only across 5 seeds × 5 VLMs.

In [4]:
from evaluation.analysis.tables import build_ablation_table
abl = build_ablation_table(REPO / "docs" / "tables" / "ablation.tex")
abl = abl.copy()
for col in abl.columns:
    abl[col] = (abl[col] * 100).round(2)
abl

metric,accuracy,aucpr,auroc,brier,ece,f1,precision,recall,sensitivity,specificity
variant,,,,,,,,,,
bce_only,68.20,85.48,75.25,19.91,9.15,74.94,74.95,75.25,75.25,55.96
full,71.52,87.43,78.61,18.43,7.08,76.86,79.39,74.82,74.82,65.15
no_brier,70.58,87.10,78.02,19.04,8.48,75.41,80.48,71.31,71.31,69.14
no_rank,68.63,85.52,75.31,19.63,8.13,75.74,74.17,77.66,77.66,52.05


## Table 4 — Null-image ablation (Qwen3-VL-8B)

Five null variants vs. correctness; black is the BICR baseline.

In [5]:
from evaluation.analysis.tables import build_null_table
null = build_null_table(REPO / "docs" / "tables" / "null_ablation.tex")
null = null.copy()
for col in null.columns:
    null[col] = (null[col] * 100).round(2)
null

metric,accuracy,aucpr,auroc,brier,ece,f1,precision,recall,sensitivity,specificity
null_type,,,,,,,,,,
black,72.81,90.14,80.08,17.47,8.86,78.66,85.38,72.97,72.97,72.46
blurred,69.19,82.40,70.07,21.53,12.55,79.92,71.02,91.54,91.54,23.73
gaussian_noise,69.61,81.57,70.35,21.25,12.31,79.73,72.14,89.22,89.22,29.71
pixel_shuffled,69.57,83.96,72.15,21.01,12.06,79.95,71.62,90.53,90.53,26.92
white,69.08,80.56,67.80,22.23,13.32,79.53,71.50,89.62,89.62,27.29


## Appendix — Per-VLM AUROC breakdown

In [6]:
from evaluation.analysis.tables import build_per_vlm_table
per_vlm = build_per_vlm_table(REPO / "docs" / "tables" / "per_vlm_auroc.tex", metric="auroc")
per_vlm = per_vlm.copy()
from evaluation.analysis.aggregate import VLMS
for v in VLMS:
    per_vlm[v] = (per_vlm[v] * 100).round(2)
per_vlm

vlm,method,method_key,InternVL3_5-14B-HF,Qwen3-VL-8B-Instruct,deepseek-vl2,gemma-3-27b-it,llava-v1.6-vicuna-13b-hf
0,P(True),PTRUE,59.45,54.57,52.64,56.79,54.29
1,Self-Probing,SELF_PROBING,70.60,59.50,61.57,67.61,66.04
2,PE,PE,43.64,52.85,73.45,61.78,68.10
3,P(IK),PIK,73.78,77.12,78.45,76.08,77.31
4,SAPLMA,SAPLMA,65.41,73.96,77.15,75.45,72.47
5,InternalInspector,II,69.31,79.53,79.31,74.15,71.86
6,CCPS,CCPS,58.22,44.87,70.95,68.47,72.88
7,BICR,BICR,76.40,80.08,81.09,76.56,78.93


## Figure 1 — Cross-VLM reliability diagram

In [7]:
from evaluation.analysis.plots import plot_cross_vlm_calibration
out = plot_cross_vlm_calibration(REPO / "docs" / "figures_generated" / "calibration_cross_vlm.pdf")
print(f"Wrote {out}")

from IPython.display import IFrame
IFrame(str(out.relative_to(REPO)), width=550, height=550)

Wrote /home/rkhanm1/VLMCE/repo/docs/figures_generated/calibration_cross_vlm.pdf
